# Notebook 1 – WHO Adult Obesity Data

## Objective
Download, clean, quality check, and export WHO age-standardized adult obesity prevalence data for selected Pacific Island countries and territories.

### Source
WHO Global Health Observatory (accessed through the Our World in Data data service)

### Study Population
Initial study population: 19 Pacific Island countries and territories

Final analytical dataset: 16 jurisdictions with complete WHO obesity data

### Output
processed_data/who_obesity_pacific_16jurisdictions_1980_2024.csv

## 1. Initialize project and download source data

In [7]:
import pandas as pd
import requests
from io import StringIO
import os

os.makedirs("raw_data/WHO", exist_ok=True)
os.makedirs("processed_data", exist_ok=True)

url = "https://ourworldindata.org/grapher/obesity-prevalence-adults-who-gho.csv"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
print("Status code:", response.status_code)

response.raise_for_status()

who_obesity_raw = pd.read_csv(StringIO(response.text))

who_obesity_raw.to_csv("raw_data/WHO/who_obesity_owid_raw.csv", index=False)

print("Rows downloaded:", len(who_obesity_raw))
print("Columns:")
print(who_obesity_raw.columns.tolist())

who_obesity_raw.head()

Status code: 200
Rows downloaded: 9270
Columns:
['Entity', 'Code', 'Year', 'Obesity among adults, BMI >= 30 kg/m2 (age-standardized estimate) (%) - Sex: both sexes - Age group: 18+  years of age']


,Entity,Code,Year,"Obesity among adults, BMI >= 30 kg/m2 (age-standardized estimate) (%) - Sex: both sexes - Age group: 18+ years of age"
0,Afghanistan,AFG,1980,0.974846
1,Afghanistan,AFG,1981,1.043918
2,Afghanistan,AFG,1982,1.119849
3,Afghanistan,AFG,1983,1.203083
4,Afghanistan,AFG,1984,1.294184


## 2. Download WHO obesity data

Download the complete WHO adult obesity dataset and save the raw file exactly as received.

In [8]:
pacific_iso3 = [
    "ASM", "COK", "FJI", "FSM", "PYF",
    "GUM", "KIR", "MHL", "NRU", "NIU",
    "MNP", "PLW", "WSM", "SLB", "TKL",
    "TON", "TUV", "VUT", "NCL"
]

value_column = [c for c in who_obesity_raw.columns if "Obesity among adults" in c][0]

obesity_pacific = who_obesity_raw[
    who_obesity_raw["Code"].isin(pacific_iso3)
].copy()

obesity_pacific = obesity_pacific.rename(columns={
    "Entity": "Country",
    "Code": "ISO3",
    value_column: "Obesity_Pct"
})

obesity_pacific = obesity_pacific[
    ["ISO3", "Country", "Year", "Obesity_Pct"]
]

print(obesity_pacific.head(20))

    ISO3         Country  Year  Obesity_Pct
180  ASM  American Samoa  1980    52.256046
181  ASM  American Samoa  1981    53.427460
182  ASM  American Samoa  1982    54.613544
183  ASM  American Samoa  1983    55.793793
184  ASM  American Samoa  1984    56.962180
185  ASM  American Samoa  1985    58.098890
186  ASM  American Samoa  1986    59.198030
187  ASM  American Samoa  1987    60.257977
188  ASM  American Samoa  1988    61.269455
189  ASM  American Samoa  1989    62.218540
190  ASM  American Samoa  1990    63.111027
191  ASM  American Samoa  1991    63.937390
192  ASM  American Samoa  1992    64.687645
193  ASM  American Samoa  1993    65.361145
194  ASM  American Samoa  1994    65.963326
195  ASM  American Samoa  1995    66.505460
196  ASM  American Samoa  1996    66.996290
197  ASM  American Samoa  1997    67.440790
198  ASM  American Samoa  1998    67.847160
199  ASM  American Samoa  1999    68.225090


## 3. Filter to Pacific Island jurisdictions

Subset the dataset to the selected Pacific Island countries and territories and keep only the variables required for analysis.

In [9]:
expected = {
    "ASM":"American Samoa",
    "COK":"Cook Islands",
    "FJI":"Fiji",
    "FSM":"Federated States of Micronesia",
    "PYF":"French Polynesia",
    "GUM":"Guam",
    "KIR":"Kiribati",
    "MHL":"Marshall Islands",
    "NRU":"Nauru",
    "NIU":"Niue",
    "MNP":"Northern Mariana Islands",
    "PLW":"Palau",
    "WSM":"Samoa",
    "SLB":"Solomon Islands",
    "TKL":"Tokelau",
    "TON":"Tonga",
    "TUV":"Tuvalu",
    "VUT":"Vanuatu",
    "NCL":"New Caledonia"
}

present = set(obesity_pacific["ISO3"].unique())

missing = {k:v for k,v in expected.items() if k not in present}

print("Present:", len(present))
print("Missing:", len(missing))
print(missing)

Present: 16
Missing: 3
{'GUM': 'Guam', 'MNP': 'Northern Mariana Islands', 'NCL': 'New Caledonia'}


## 4. Quality assurance

Verify that the cleaned dataset contains the expected countries, years, and observations.

In [10]:
print(obesity_pacific.info())

print()

print("Countries:")
print(sorted(obesity_pacific["Country"].unique()))

print()

print("Years:")
print(obesity_pacific["Year"].min(), "to", obesity_pacific["Year"].max())

<class 'pandas.core.frame.DataFrame'>
Index: 720 entries, 180 to 8954
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ISO3         720 non-null    object 
 1   Country      720 non-null    object 
 2   Year         720 non-null    int64  
 3   Obesity_Pct  720 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 28.1+ KB
None

Countries:
['American Samoa', 'Cook Islands', 'Fiji', 'French Polynesia', 'Kiribati', 'Marshall Islands', 'Micronesia (country)', 'Nauru', 'Niue', 'Palau', 'Samoa', 'Solomon Islands', 'Tokelau', 'Tonga', 'Tuvalu', 'Vanuatu']

Years:
1980 to 2024


## 5. Coverage assessment

Determine which jurisdictions have complete obesity data and identify those excluded due to missing observations.

In [11]:
coverage = (
    obesity_pacific
    .groupby(["ISO3", "Country"])
    .agg(
        First_Year=("Year", "min"),
        Last_Year=("Year", "max"),
        Years=("Year", "count")
    )
    .reset_index()
)

coverage

,ISO3,Country,First_Year,Last_Year,Years
0,ASM,American Samoa,1980,2024,45
1,COK,Cook Islands,1980,2024,45
2,FJI,Fiji,1980,2024,45
3,FSM,Micronesia (country),1980,2024,45
4,KIR,Kiribati,1980,2024,45
5,MHL,Marshall Islands,1980,2024,45
6,NIU,Niue,1980,2024,45
7,NRU,Nauru,1980,2024,45
8,PLW,Palau,1980,2024,45
9,PYF,French Polynesia,1980,2024,45


## 6. Export cleaned dataset

Save the cleaned analytical dataset and coverage table for use in subsequent notebooks.

In [12]:
obesity_pacific.to_csv(
    "processed_data/adult_obesity.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


# Notebook Summary

This notebook successfully:

- Downloaded WHO adult obesity data
- Filtered to Pacific Island jurisdictions
- Identified 16 jurisdictions with complete coverage
- Excluded Guam, Northern Mariana Islands, and New Caledonia due to missing WHO obesity data
- Exported a clean dataset for downstream analysis

This dataset serves as the primary obesity outcome variable for the Pacific Food Import Dependency project.